Load the transformed data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [2]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-Optimization")
    .master("local[*]")
    .getOrCreate()
)

In [3]:
transformed_path = "../data/processed/support_tickets_transformed"

In [4]:
parquet_files = [
    os.path.join(transformed_path, file)
    for file in os.listdir(transformed_path)
    if file.endswith(".parquet")
]

In [5]:
tickets_df = spark.read.parquet(
    *parquet_files
)

print("Data loaded successfully.")

Data loaded successfully.


Check the Number of Partitions

In [6]:
tickets_df.rdd.getNumPartitions()

8

repartition()

In [7]:
repartitioned_df = tickets_df.repartition(8)

In [8]:
print(
    "Partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Partitions: 8


coalesce()

In [9]:
coalesced_df = tickets_df.coalesce(2)

In [10]:
print(
    "Partitions:",
    coalesced_df.rdd.getNumPartitions()
)

Partitions: 2


Cache

In [11]:


cached_df = tickets_df.cache()

cached_df.count()

print(
    "Is cached:",
    cached_df.is_cached
)

Is cached: True


Reuse the cached DataFrame

In [12]:
cached_df.groupBy(
    "category"
).count().show()


+---------------+------+
|       category| count|
+---------------+------+
|        Payroll|201174|
|        Core HR|200461|
|    Recruitment|200888|
|Time Management|200438|
|    Performance|200394|
|       Benefits|200737|
+---------------+------+



In [13]:
cached_df.groupBy(
    "priority"
).count().show()

+--------+------+
|priority| count|
+--------+------+
|    High|289795|
|     Low|275946|
| Unknown| 80905|
|  Medium|276597|
|Critical|280849|
+--------+------+



Remove the cache

In [14]:
cached_df.unpersist()

DataFrame[ticket_id: string, customer_id: string, agent_id: string, created_date: date, closed_date: date, priority: string, category: string, sub_category: string, product: string, status: string, channel: string, issue_description: string, resolution: string, customer_region: string, first_response_minutes: int, resolution_hours: double, customer_satisfaction: double, escalation_flag: string, created_year: int, created_month: int, resolution_category: string, first_response_category: string, satisfaction_category: string, escalation_indicator: int, priority_level: int]

In [15]:
print(
    "Is DataFrame cached:",
    cached_df.is_cached
)

Is DataFrame cached: False


persist()

In [17]:
from pyspark import StorageLevel

In [18]:
persisted_df = tickets_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

In [21]:
persisted_df.count()

print(
    "Is DataFrame cached:",
    persisted_df.is_cached
)

Is DataFrame cached: True


Remove persistence

In [22]:
persisted_df.unpersist()

DataFrame[ticket_id: string, customer_id: string, agent_id: string, created_date: date, closed_date: date, priority: string, category: string, sub_category: string, product: string, status: string, channel: string, issue_description: string, resolution: string, customer_region: string, first_response_minutes: int, resolution_hours: double, customer_satisfaction: double, escalation_flag: string, created_year: int, created_month: int, resolution_category: string, first_response_category: string, satisfaction_category: string, escalation_indicator: int, priority_level: int]

Lazy Evaluation

In [23]:
high_priority_df = tickets_df.filter(
    F.col("priority").isin(
        ["High", "Critical"]
    )
)

print("Transformation created.")

Transformation created.


In [ ]:
high_priority_count = high_priority_df.count()

print(
    "High/Critical priority tickets:",
    high_priority_count
)

## Spark Optimization Summary

### Partitions
Partitions divide large datasets into smaller chunks that Spark can process in parallel.

### repartition()
Used to redistribute data into a specified number of partitions.
It generally causes a shuffle.

### coalesce()
Used mainly to reduce the number of partitions with less data movement.

### cache()
Stores a DataFrame for reuse.

### persist()
Stores a DataFrame using a selected storage level.

### Lazy Evaluation
Spark delays execution of transformations until an action is performed.

### Transformations
Examples:
- filter()
- select()
- withColumn()
- join()

### Actions
Examples:
- show()
- count()
- collect()
- write()

### collect()
Should be avoided on large datasets because it brings all rows to the driver.